# Regression Trees & Random Forest Regression

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
from sklearn import tree
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.datasets import make_regression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Import the apartment data

In [ ]:
# Define columns for import
columns = [ 'web-scraper-order',
            'address_raw',
            'rooms',
            'area',
            'luxurious',
            'price',
            'price_per_m2',
            'lat',
            'lon',
            'bfs_number',
            'bfs_name',
            'pop',
            'pop_dens',
            'frg_pct',
            'emp',
            'mean_taxable_income',
            'dist_supermarket']

# Read and select variables
df_orig = pd.read_csv("apartments_data_enriched_cleaned.csv", sep=";", encoding='utf-8')[columns]

# Rename variable 'web-scraper-order' to 'apmt_id'
df_orig = df_orig.rename(columns={'web-scraper-order': 'id'})

# Remove missing values
df = df_orig.dropna()
df.head(5)

# Remove duplicates
df = df.drop_duplicates()

# Remove some 'extreme' values
df = df.loc[(df['price'] >= 1000) & 
            (df['price'] <= 5000)]

# Reset index
df = df.reset_index(drop=True)

print(df.shape)
df.head(5)

## Regression Tree
See also: https://data36.com/regression-tree-python-scikit-learn

### Create train and test samples for the regression tree (train = 80%, test = 20% of the data)

In [ ]:
# Create train and test samples
X_train, X_test, y_train, y_test = train_test_split(df[['area', 
                                                        'rooms',
                                                        'pop_dens',
                                                        'mean_taxable_income',
                                                        'dist_supermarket']], 
                                                        df['price'], 
                                                        test_size=0.20, 
                                                        random_state=42)

# Show X_train
print('X_train:')
print(X_train.head(), '\n')

# Show y_train
print('y_train:')
print(y_train.head())

### Fit the regression tree model

In [ ]:
# Create decision tree regressor object
reg = DecisionTreeRegressor(random_state=20, max_depth=3)

# Train decision tree regressor
reg = reg.fit(X_train, y_train)

# Predict the response for test dataset
y_pred = reg.predict(X_test)

### Calculate coefficient of determination (R-squared)

In [ ]:
# Calculate coefficient of determination, round to 4 decimals
print(f'R-squared:, {r2_score(y_test, y_pred):.4f}')

### Print text representation of the regression tree

In [ ]:
# Text representation of the regression tree
text_representation = tree.export_text(reg, 
                                       feature_names=list(X_train.columns))

# Print text_representation
print(text_representation)

## Task 2b: Changing max_depth to 5

In [ ]:
# Create decision tree regressor object with max_depth=5
reg_depth5 = DecisionTreeRegressor(random_state=20, max_depth=5)

# Train decision tree regressor
reg_depth5 = reg_depth5.fit(X_train, y_train)

# Predict the response for test dataset
y_pred_depth5 = reg_depth5.predict(X_test)

# Calculate coefficient of determination
print(f'R-squared (max_depth=5): {r2_score(y_test, y_pred_depth5):.4f}')

# Text representation of the regression tree
text_representation_depth5 = tree.export_text(reg_depth5, 
                                              feature_names=list(X_train.columns))
print('\nText representation of the regression tree (max_depth=5):')
print(text_representation_depth5)

# Visualize the regression tree
fig = plt.figure(figsize=(16,8))
_ = tree.plot_tree(reg_depth5, 
                   feature_names=list(X_train.columns),  
                   class_names=['price'],
                   filled=True,
                   fontsize=7,
                   label='root',
                   rounded=True)

### Explanation (Task 2b):
With max_depth=5, the regression tree becomes more complex with more nodes and splits compared to max_depth=3. This results in a more detailed tree that can capture more nuanced patterns in the data. The text representation shows more levels of decision rules, and the visualization displays a larger, more intricate tree structure. The R-squared value typically increases with greater depth as the model fits the training data more closely, though this can lead to overfitting.

## Task 2c: Dropping area and rooms variables

In [ ]:
# Create train and test samples WITHOUT area and rooms
X_train_no_ar, X_test_no_ar, y_train_no_ar, y_test_no_ar = train_test_split(df[['pop_dens',
                                                                                  'mean_taxable_income',
                                                                                  'dist_supermarket']], 
                                                                              df['price'], 
                                                                              test_size=0.20, 
                                                                              random_state=42)

# Create decision tree regressor object
reg_no_ar = DecisionTreeRegressor(random_state=20, max_depth=3)

# Train decision tree regressor
reg_no_ar = reg_no_ar.fit(X_train_no_ar, y_train_no_ar)

# Predict the response for test dataset
y_pred_no_ar = reg_no_ar.predict(X_test_no_ar)

# Calculate coefficient of determination
print(f'R-squared (without area and rooms): {r2_score(y_test_no_ar, y_pred_no_ar):.4f}')
print(f'R-squared (with area and rooms):    {r2_score(y_test, y_pred):.4f}')

### Explanation (Task 2c):
Yes, the R-squared value changes significantly when dropping the area and rooms variables. The R-squared decreases substantially because area and rooms are among the most important predictors of apartment prices. These variables directly relate to the size and capacity of an apartment, which are strong determinants of price. Without these features, the model has less predictive power and can only rely on location-based and demographic variables (pop_dens, mean_taxable_income, dist_supermarket), which explain less of the variance in apartment prices.

### Vizualizing the regression tree

In [ ]:
fig = plt.figure(figsize=(12,6))
_ = tree.plot_tree(reg, 
                   feature_names=list(X_train.columns),  
                   class_names=['price'],
                   filled=True,
                   fontsize=9,
                   label='root',
                   rounded=True)

## Random Forest Regression
For details see: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

### Create train and test samples for the random forest (train = 80%, test = 20% of the data)

In [ ]:
# Create train and test samples (the names X2_ and y2_ were used because X_ and y_ were already used above)
X2_train, X2_test, y2_train, y2_test = train_test_split(df[['area', 
                                                            'rooms',
                                                            'pop_dens',
                                                            'mean_taxable_income',
                                                            'dist_supermarket']], 
                                                            df['price'], 
                                                            test_size=0.20, 
                                                            random_state=42)

# Show X2_train
print('X2_train:')
print(X2_train.head(), '\n')

# Show y2_train
print('y2_train:')
print(y2_train.head())

### Fit the Random Forest Regression

In [ ]:
X, y = make_regression(n_features=4, n_informative=2,
                       random_state=5, shuffle=False)


reg_rf = RandomForestRegressor(n_estimators=500, 
                               max_depth=10, 
                               random_state=5)
reg_rf.fit(X2_train, y2_train)

# Calculate coefficient of determination (R-squared)
print(f'R-squared: {reg_rf.score(X2_test, y2_test):.4f}')

### Show feature importance

In [ ]:
cols = X2_train.columns

# Derive feature importance from random forest
importances = reg_rf.feature_importances_
std         = np.std([tree.feature_importances_ for tree in reg_rf.estimators_], axis=0)
indices     = np.argsort(importances)[::-1]

# Print col-names and importances-values
print( cols[indices] )
print( importances[indices] )

# Barplot with feature importance
df_fi = pd.DataFrame({'features':cols,'importances': importances})
df_fi.sort_values('importances', inplace=True)
df_fi.plot(kind='barh', 
           y='importances', 
           x='features', 
           color='darkred', 
           figsize=(6,3))

plt.show()

## Task 2e: Random Forest without area variable

In [ ]:
# Create train and test samples WITHOUT area variable
X3_train, X3_test, y3_train, y3_test = train_test_split(df[['rooms',
                                                            'pop_dens',
                                                            'mean_taxable_income',
                                                            'dist_supermarket']], 
                                                            df['price'], 
                                                            test_size=0.20, 
                                                            random_state=42)

# Fit the Random Forest Regression without area
reg_rf_no_area = RandomForestRegressor(n_estimators=500, 
                                       max_depth=10, 
                                       random_state=5)
reg_rf_no_area.fit(X3_train, y3_train)

# Calculate coefficient of determination (R-squared)
print(f'R-squared (without area): {reg_rf_no_area.score(X3_test, y3_test):.4f}')
print(f'R-squared (with area):    {reg_rf.score(X2_test, y2_test):.4f}')

# Show feature importance for model without area
cols_no_area = X3_train.columns
importances_no_area = reg_rf_no_area.feature_importances_
indices_no_area = np.argsort(importances_no_area)[::-1]

print('\nFeature importances (without area):')
print(cols_no_area[indices_no_area])
print(importances_no_area[indices_no_area])

# Barplot with feature importance
df_fi_no_area = pd.DataFrame({'features':cols_no_area,'importances': importances_no_area})
df_fi_no_area.sort_values('importances', inplace=True)
df_fi_no_area.plot(kind='barh', 
                   y='importances', 
                   x='features', 
                   color='darkred', 
                   figsize=(6,3))
plt.title('Feature Importance (without area)')
plt.show()

### Explanation (Task 2f):
Yes, the importance of features changes when the area variable is dropped from the model. In the original model with area included, area was by far the most important feature (highest importance value), followed by pop_dens, dist_supermarket, mean_taxable_income, and rooms. 

When area is removed from the model:
1. The rooms variable becomes more important because it serves as a proxy for apartment size (rooms and area are correlated).
2. The other features (pop_dens, mean_taxable_income, dist_supermarket) also see their relative importance increase to compensate for the missing information from the area variable.
3. The overall R-squared of the model decreases because area was such a strong predictor, and the remaining variables cannot fully compensate for its absence.

This demonstrates that feature importance is context-dependent and can shift when the feature set changes, especially when removing a highly correlated and important feature like area.

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')